# LeetCode 3827 — Count Monobit Integers bằng Python

Link đề: https://leetcode.com/problems/count-monobit-integers/ · Độ khó: **Easy** · Chủ đề: Bit Manipulation, Math, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def countMonobit(self, n: int) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Weekly Contest 487, câu 1, Easy, không Premium).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu thế nào là số **monobit**: biểu diễn nhị phân của nó chỉ gồm toàn bit `0` hoặc toàn bit `1`.
2. Biết đổi số sang nhị phân bằng `bin(x)`, `x.bit_length()` và các phép dịch bit `<<`, `>>`.
3. Nhận ra các số monobit chỉ có dạng `0` và `2^k - 1` (`1, 3, 7, 15, ...`), từ đó viết lời giải `O(log n)` và thậm chí `O(1)` bằng công thức.
4. Viết 5 cách khác nhau (duyệt + chuỗi nhị phân, mẹo bit `x & (x+1)`, sinh trực tiếp, so sánh với mặt nạ, công thức) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi quen thuộc: quên số `0`, đếm nhầm `2^k` thay vì `2^k - 1`, sai biên khi `n` đúng bằng `2^k - 1`.

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Một số nguyên không âm được gọi là **Monobit** nếu **mọi bit** trong biểu diễn nhị phân của nó đều giống nhau (toàn `0` hoặc toàn `1`). Hãy trả về **số lượng** số Monobit trong đoạn `[0, n]` (tính cả hai đầu).

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `1` | `2` | Đoạn `[0, 1]` có `"0"` và `"1"`, cả hai đều đồng nhất bit |
| 2 | `4` | `3` | `0`, `1`, `10`, `11`, `100`: chỉ `0`, `1`, `3` (`"11"`) là Monobit |

Chú ý: số `0` có biểu diễn nhị phân `"0"` (một bit) nên **được tính** là Monobit.

Constraints: `0 <= n <= 1000`.

In [1]:
for n in [0, 1, 4, 7, 10]:
    row = [(x, bin(x)[2:]) for x in range(n + 1)]
    print(f"n={n}: ", ' '.join(f"{x}={b}" for x, b in row))

n=0:  0=0
n=1:  0=0 1=1
n=4:  0=0 1=1 2=10 3=11 4=100
n=7:  0=0 1=1 2=10 3=11 4=100 5=101 6=110 7=111
n=10:  0=0 1=1 2=10 3=11 4=100 5=101 6=110 7=111 8=1000 9=1001 10=1010


## 3. Kiến thức nền

### 3.1. Biểu diễn nhị phân trong Python

`bin(x)` trả về chuỗi dạng `'0b101'`. Cắt hai ký tự đầu bằng `[2:]` để lấy dãy bit. `bin(0)` là `'0b0'`, tức dãy bit `"0"`.

In [2]:
for x in [0, 1, 3, 4, 5, 7]:
    print(x, bin(x), bin(x)[2:])

0 0b0 0
1 0b1 1
3 0b11 11
4 0b100 100
5 0b101 101
7 0b111 111


### 3.2. `set` để kiểm tra "mọi phần tử giống nhau"

`set("1011")` ra `{'1', '0'}`. Một chuỗi có mọi ký tự giống nhau khi và chỉ khi `len(set(chuỗi)) == 1`.

In [3]:
for b in ["0", "111", "1011", "1000"]:
    print(b, set(b), len(set(b)) == 1)

0 {'0'} True
111 {'1'} True
1011 {'0', '1'} False
1000 {'0', '1'} False


### 3.3. `bit_length`, dịch bit và `&`

- `x.bit_length()`: số bit cần để biểu diễn `x` (với `x = 0` là `0`).
- `1 << k` bằng `2^k`; `(1 << k) - 1` là số có `k` bit `1` liên tiếp (`k = 3` cho `111` = 7).
- `a & b`: AND từng bit. Nếu `x = 0b0111` thì `x + 1 = 0b1000` và `x & (x + 1) = 0`.

In [4]:
for x in [0, 1, 3, 4, 7, 12]:
    print(f"x={x:2d} {bin(x):>8} bit_length={x.bit_length()} x&(x+1)={x & (x + 1)}")
print([(1 << k) - 1 for k in range(0, 6)])

x= 0      0b0 bit_length=0 x&(x+1)=0
x= 1      0b1 bit_length=1 x&(x+1)=0
x= 3     0b11 bit_length=2 x&(x+1)=0
x= 4    0b100 bit_length=3 x&(x+1)=4
x= 7    0b111 bit_length=3 x&(x+1)=0
x=12   0b1100 bit_length=4 x&(x+1)=12
[0, 1, 3, 7, 15, 31]


### 3.4. Vòng lặp `while` với điều kiện kết thúc

Khi không biết trước số lần lặp ta dùng `while điều_kiện:`. Ở bài này ta lặp "tạo thêm số dạng `2^k - 1` chừng nào chưa vượt `n`".

In [5]:
n = 20
cur, count = 1, 0
while cur <= n:
    count += 1
    print("lấy", cur, bin(cur))
    cur = cur * 2 + 1       # 1 -> 3 -> 7 -> 15 -> 31
print("số lượng:", count)

lấy 1 0b1
lấy 3 0b11
lấy 7 0b111
lấy 15 0b1111
số lượng: 4


### 3.5. `sum` với biểu thức điều kiện và `range`

`range(n + 1)` sinh `0, 1, ..., n` (không tính `n + 1`), nên để duyệt **cả hai đầu** của `[0, n]` ta viết `range(n + 1)`. `sum(1 for ... if ...)` là cách ngắn gọn để đếm số phần tử thỏa điều kiện: mỗi phần tử thỏa góp thêm `1`.

In [6]:
n = 10
print(list(range(n + 1)))
print(sum(1 for x in range(n + 1) if x % 2 == 0))      # đếm số chẵn trong [0, 10]
print(sum(1 for x in range(n + 1) if x & (x + 1) == 0))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
6
4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Liệt kê nhị phân của vài số đầu tiên và tô những số có bit đồng nhất:

| x | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 15 |
|---|---|---|---|---|---|---|---|---|---|---|
| nhị phân | 0 | 1 | 10 | 11 | 100 | 101 | 110 | 111 | 1000 | 1111 |
| Monobit? | có | có | không | có | không | không | không | có | không | có |

Quan sát: ngoài `0`, bit đầu tiên (bit cao nhất) của mọi số dương luôn là `1`. Vậy số dương Monobit phải **toàn bit `1`**, tức là dạng `1`, `11`, `111`, ... = `1, 3, 7, 15, ...` = `2^k - 1`.

Với `n = 4`: `0` (đặc biệt), `1`, `3` -> 3 số. Với `n = 7`: `0, 1, 3, 7` -> 4 số. Với `n = 10`: `0, 1, 3, 7` -> vẫn 4 (vì `15 > 10`).

In [7]:
def is_monobit_text(x):
    return len(set(bin(x)[2:])) == 1

mono = [x for x in range(40) if is_monobit_text(x)]
print(mono)
print([(1 << k) - 1 for k in range(0, 6)])

[0, 1, 3, 7, 15, 31]
[0, 1, 3, 7, 15, 31]


### Vì sao bit đầu luôn là 1?

Biểu diễn nhị phân chuẩn **không có số 0 thừa ở đầu**: `5` viết là `101` chứ không phải `0101`. Vì vậy bit đầu tiên của mọi số dương luôn là `1`. Nếu mọi bit phải giống bit đầu thì tất cả đều là `1`. Điều này loại ngay các số như `2 = 10`, `4 = 100`, `5 = 101`, `6 = 110`. Chỉ còn các số "toàn một" `1, 11, 111, ...`.

Thử đoán rồi kiểm chứng bằng code: trong `[0, 1000]` có bao nhiêu số Monobit? Các số `2^k - 1` nhỏ hơn hoặc bằng 1000 là `1, 3, 7, 15, 31, 63, 127, 255, 511` (9 số), cộng số `0` thành 10.

In [8]:
mono = [x for x in range(1001) if len(set(bin(x)[2:])) == 1]
print(mono, len(mono))
assert len(mono) == 10

[0, 1, 3, 7, 15, 31, 63, 127, 255, 511] 10


## 5. Cách 1 — Brute Force / baseline

Duyệt `x` từ `0` đến `n`, đổi sang chuỗi nhị phân, kiểm tra `len(set(...)) == 1`, đếm. Đây đúng bằng định nghĩa của đề.

Nút thắt: mỗi số tốn `O(log x)` để đổi nhị phân và kiểm tra, tổng `O(n log n)`. Với `n <= 1000` vẫn rất nhanh nhưng ta đang lãng phí: hầu hết số bị loại.

In [9]:
class SolutionBrute:
    def countMonobit(self, n):
        count = 0
        for x in range(n + 1):
            bits = bin(x)[2:]
            if len(set(bits)) == 1:
                count += 1
        return count

print(SolutionBrute().countMonobit(1), SolutionBrute().countMonobit(4), SolutionBrute().countMonobit(0))

2 3 1


**Độ phức tạp:** Time `O(n log n)`, Space `O(log n)` cho chuỗi nhị phân.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** số dương Monobit chính là `2^k - 1`. Có hai cách nhìn tương đương:

- **Dạng đóng:** `x + 1` là lũy thừa của 2 (`1000...0`), nên `x & (x + 1) == 0`. Đúng cả với `x = 0` (vì `0 & 1 = 0`), nên mẹo này gộp luôn số `0`.
- **Mặt nạ:** `x == (1 << x.bit_length()) - 1`: `x` bằng số toàn bit `1` cùng độ dài. Với `x = 0`: `bit_length = 0`, mặt nạ `= 0` nên cũng thỏa.

Quy tắc "nếu... thì...": nếu `x & (x + 1) == 0` thì Monobit; ngược lại thì không. Từ đó ta chỉ cần **sinh** các số `2^k - 1` nhỏ hơn hoặc bằng `n` thay vì duyệt mọi số. Số lượng chúng là `bit_length(n+1)`: có công thức `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — Mẹo bit `x & (x + 1) == 0`

Duyệt `x` từ `0` đến `n` nhưng kiểm tra bằng một phép AND, không tạo chuỗi.

In [10]:
class SolutionBitTrick:
    def countMonobit(self, n):
        count = 0
        for x in range(n + 1):
            if x & (x + 1) == 0:        # x có dạng 0 hoặc 2^k - 1
                count += 1
        return count

print(SolutionBitTrick().countMonobit(1), SolutionBitTrick().countMonobit(4))

2 3


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — Sinh trực tiếp các số `2^k - 1`

Bắt đầu với đáp án `1` (cho số `0`), `cur = 1`; chừng nào `cur <= n` thì tăng đáp án và `cur = cur * 2 + 1` (thêm một bit `1`).

In [11]:
class SolutionBuild:
    def countMonobit(self, n):
        count = 1               # số 0
        cur = 1                 # 1, 11, 111, ... trong hệ nhị phân
        while cur <= n:
            count += 1
            cur = cur * 2 + 1
        return count

print(SolutionBuild().countMonobit(1), SolutionBuild().countMonobit(4), SolutionBuild().countMonobit(0))

2 3 1


**Độ phức tạp:** Time `O(log n)` (số lần lặp bằng số bit của `n`), Space `O(1)`.

### Cách 4 — So sánh với mặt nạ `(1 << bit_length) - 1`

Với mỗi `x`, mặt nạ toàn bit `1` cùng độ dài là `(1 << x.bit_length()) - 1`. `x` là Monobit khi bằng mặt nạ này. Ta dùng cách này khi muốn "đọc" được ý nghĩa: **mọi bit đều là 1**.

In [12]:
class SolutionMask:
    def countMonobit(self, n):
        count = 0
        for x in range(n + 1):
            mask = (1 << x.bit_length()) - 1     # x = 0 -> mask = 0
            if x == mask:
                count += 1
        return count

print(SolutionMask().countMonobit(1), SolutionMask().countMonobit(4))

2 3


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 5 — Công thức `O(1)`: `(n + 1).bit_length()`

Các số Monobit trong `[0, n]` là `0` và `2^k - 1` với `1 <= k` và `2^k - 1 <= n`, tức `2^k <= n + 1`, tức `k <= floor(log2(n + 1))`. `floor(log2(n+1)) = (n+1).bit_length() - 1`. Cộng thêm số `0` được `(n + 1).bit_length()`.

In [13]:
class SolutionFormula:
    def countMonobit(self, n):
        return (n + 1).bit_length()

print([SolutionFormula().countMonobit(n) for n in [0, 1, 2, 3, 4, 7, 8, 15, 1000]])

[1, 2, 2, 3, 3, 4, 4, 5, 10]


**Độ phức tạp:** Time `O(1)` (một phép `bit_length`), Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 với `n = 10`:

| Vòng | cur (nhị phân) | `cur <= 10`? | count |
|---|---|---|---|
| khởi tạo | 1 (1) | | 1 (số 0) |
| 1 | 1 (1) | có | 2 |
| 2 | 3 (11) | có | 3 |
| 3 | 7 (111) | có | 4 |
| 4 | 15 (1111) | không, dừng | 4 |

In [14]:
def trace_build(n):
    count = 1
    cur = 1
    print(f"khởi tạo: count={count} (số 0), cur={cur} ({bin(cur)[2:]})")
    step = 0
    while cur <= n:
        step += 1
        count += 1
        print(f"vòng {step}: cur={cur} ({bin(cur)[2:]}) <= {n} -> count={count}")
        cur = cur * 2 + 1
    print(f"cur={cur} ({bin(cur)[2:]}) > {n}: dừng, đáp án = {count}")
    return count

assert trace_build(10) == 4
assert trace_build(4) == 3

khởi tạo: count=1 (số 0), cur=1 (1)
vòng 1: cur=1 (1) <= 10 -> count=2
vòng 2: cur=3 (11) <= 10 -> count=3
vòng 3: cur=7 (111) <= 10 -> count=4
cur=15 (1111) > 10: dừng, đáp án = 4
khởi tạo: count=1 (số 0), cur=1 (1)
vòng 1: cur=1 (1) <= 4 -> count=2
vòng 2: cur=3 (11) <= 4 -> count=3
cur=7 (111) > 4: dừng, đáp án = 3


In [15]:
def trace_bit_trick(n):
    for x in range(n + 1):
        a = x & (x + 1)
        print(f"x={x:2d} {bin(x)[2:]:>5}  x+1={x+1:2d} {bin(x+1)[2:]:>5}  AND={a:2d}  -> {'Monobit' if a == 0 else 'không'}")

trace_bit_trick(8)

x= 0     0  x+1= 1     1  AND= 0  -> Monobit
x= 1     1  x+1= 2    10  AND= 0  -> Monobit
x= 2    10  x+1= 3    11  AND= 2  -> không
x= 3    11  x+1= 4   100  AND= 0  -> Monobit
x= 4   100  x+1= 5   101  AND= 4  -> không
x= 5   101  x+1= 6   110  AND= 4  -> không
x= 6   110  x+1= 7   111  AND= 6  -> không
x= 7   111  x+1= 8  1000  AND= 0  -> Monobit
x= 8  1000  x+1= 9  1001  AND= 8  -> không


## 9. Code nộp LeetCode

Cách 3 (sinh trực tiếp) hoặc Cách 5 (công thức) đều đúng; dưới đây dùng Cách 3 vì dễ giải thích và an toàn biên.

In [16]:
from typing import List, Optional


class Solution:
    def countMonobit(self, n: int) -> int:
        count = 1                  # số 0 luôn là Monobit
        cur = 1                    # 1, 3, 7, 15, ... = 2^k - 1
        while cur <= n:
            count += 1
            cur = cur * 2 + 1
        return count


print(Solution().countMonobit(1), Solution().countMonobit(4))

2 3


## 10. Vì sao thuật toán đúng

1. **Số dương Monobit phải toàn bit `1`.** Bit cao nhất của số dương luôn là `1`; nếu mọi bit giống nhau thì tất cả đều là `1`. Số `k` bit toàn `1` bằng `2^k - 1`.
2. **Số `0` là trường hợp riêng**: `"0"` toàn bit `0`, được tính một lần.
3. **Hai họ này không trùng nhau**, và mỗi `k >= 1` cho đúng một số, tăng chặt theo `k`.
4. **Vòng lặp** duyệt `cur = 1, 3, 7, ...` theo thứ tự tăng và dừng ngay khi vượt `n`, nên đếm đủ và không dư.
5. **Biên:** nếu `n = 2^k - 1` đúng thì `cur = n` vẫn thỏa `cur <= n` và được đếm (ví dụ `n = 3` cho `0, 1, 3` = 3).

## 11. Phân tích Time / Space Complexity

Gọi `n` là giá trị đầu vào, `b = bit_length(n)` khoảng `log2 n`.

- Cách 1: `O(n log n)` thời gian (mỗi số đổi sang chuỗi `O(log n)`), `O(log n)` phụ.
- Cách 2, 4: `O(n)` thời gian, `O(1)` phụ.
- Cách 3: `O(log n)` thời gian, `O(1)` phụ.
- Cách 5: `O(1)` thời gian, `O(1)` phụ.

Với `n <= 1000` mọi cách đều chạy dưới một phần nghìn giây; sự khác biệt chủ yếu để luyện tư duy bit. Không có "trường hợp xấu nhất" khác trung bình vì cả năm cách đều không phụ thuộc cấu trúc đầu vào.

## 12. Bộ test

Đối chiếu cả 5 cách với ví dụ đề, các biên (`0`, `2^k - 1`, `2^k`) và toàn bộ `n` từ `0` đến `1000`.

In [17]:
import random

solutions = [SolutionBrute(), SolutionBitTrick(), SolutionBuild(), SolutionMask(), SolutionFormula(), Solution()]

# ví dụ của đề
for s in solutions:
    assert s.countMonobit(1) == 2
    assert s.countMonobit(4) == 3

# biên
cases = {0: 1, 2: 2, 3: 3, 6: 3, 7: 4, 8: 4, 15: 5, 16: 5, 1000: 10}
for n, expected in cases.items():
    for s in solutions:
        assert s.countMonobit(n) == expected, (type(s).__name__, n)

# toàn bộ n hợp lệ
for n in range(0, 1001):
    ref = SolutionBrute().countMonobit(n)
    for s in solutions:
        assert s.countMonobit(n) == ref, (type(s).__name__, n)

# ngẫu nhiên có seed
random.seed(38270)
for _ in range(200):
    n = random.randint(0, 1000)
    assert len({s.countMonobit(n) for s in solutions}) == 1
print("tất cả test OK")

tất cả test OK


### Test thuộc tính (property test)

Ngoài đối chiếu giữa các cách, ta kiểm tra những tính chất mà đáp án **phải** thỏa: (1) đáp án không giảm theo `n`; (2) đáp án tăng thêm 1 đúng tại các `n = 2^k - 1`; (3) đáp án với `n = 2^k - 1` bằng `k + 1`.

In [18]:
prev = Solution().countMonobit(0)
jumps = []
for n in range(1, 1001):
    cur = Solution().countMonobit(n)
    assert cur >= prev
    if cur == prev + 1:
        jumps.append(n)
    else:
        assert cur == prev
    prev = cur
print("các n làm đáp án tăng:", jumps)
assert jumps == [1, 3, 7, 15, 31, 63, 127, 255, 511]
for k in range(1, 10):
    assert Solution().countMonobit((1 << k) - 1) == k + 1
print("property test OK")

các n làm đáp án tăng: [1, 3, 7, 15, 31, 63, 127, 255, 511]
property test OK


### Đo thời gian (tùy chọn)

Với `n` lớn hơn giới hạn đề, sự khác biệt giữa `O(n)` và `O(log n)` mới lộ rõ. Đoạn dưới thử `n = 200000` cho các cách duyệt và cách sinh; chỉ kiểm tra kết quả, không in thời gian để output ổn định.

In [19]:
big = 200000
r = {type(s).__name__: s.countMonobit(big) for s in [SolutionBitTrick(), SolutionBuild(), SolutionMask(), SolutionFormula()]}
print(r)
assert len(set(r.values())) == 1 and r["SolutionFormula"] == 18

{'SolutionBitTrick': 18, 'SolutionBuild': 18, 'SolutionMask': 18, 'SolutionFormula': 18}


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên số `0`

Nhiều bạn nghĩ "dãy 1, 11, 111" rồi chỉ đếm `2^k - 1` với `k >= 1`, bỏ sót `0`.

In [20]:
def wrong_forget_zero(n):
    count, cur = 0, 1               # SAI: bắt đầu từ 0 thay vì 1
    while cur <= n:
        count += 1
        cur = cur * 2 + 1
    return count

print(wrong_forget_zero(4), "!=", Solution().countMonobit(4))
assert wrong_forget_zero(4) == 2 and Solution().countMonobit(4) == 3
assert wrong_forget_zero(0) == 0 and Solution().countMonobit(0) == 1

2 != 3


### Lỗi 2: đếm lũy thừa của 2 thay vì `2^k - 1`

`1, 2, 4, 8` là lũy thừa của 2 (chỉ một bit `1`), **không phải** Monobit (trừ `1`). Cần `1, 3, 7, 15`.

In [21]:
def wrong_power_of_two(n):
    count, cur = 1, 1
    while cur <= n:
        count += 1
        cur *= 2                    # SAI: 1, 2, 4, 8 ...
    return count

print(wrong_power_of_two(4), "!=", Solution().countMonobit(4))
assert wrong_power_of_two(4) == 4 and Solution().countMonobit(4) == 3

4 != 3


### Lỗi 3: sai biên bằng `<` thay vì `<=`

Khi `n` đúng bằng `2^k - 1` (ví dụ `n = 3` hoặc `n = 7`), điều kiện `cur < n` bỏ mất chính số đó.

In [22]:
def wrong_strict(n):
    count, cur = 1, 1
    while cur < n:                  # SAI: phải là <=
        count += 1
        cur = cur * 2 + 1
    return count

print(wrong_strict(3), "!=", Solution().countMonobit(3))
assert wrong_strict(3) == 2 and Solution().countMonobit(3) == 3
assert wrong_strict(7) == 3 and Solution().countMonobit(7) == 4

2 != 3


### Lỗi 4: công thức lệch một đơn vị

Dùng `n.bit_length()` thay vì `(n + 1).bit_length()` đúng với `n = 4` nhưng sai khi `n = 2^k - 1`.

In [23]:
def wrong_formula(n):
    return n.bit_length()          # SAI: phải là (n + 1).bit_length()

print([wrong_formula(n) for n in (3, 4, 7)], [Solution().countMonobit(n) for n in (3, 4, 7)])
assert wrong_formula(3) == 2 and Solution().countMonobit(3) == 3
assert wrong_formula(4) == 3 == Solution().countMonobit(4)      # trùng hợp ở n = 4 nên dễ lọt ví dụ

[2, 3, 3] [3, 3, 4]


### Lỗi 5: tin rằng `bin(x)` không có tiền tố

`bin(5)` là `'0b101'`, trong đó có chữ `b` và số `0` ở đầu. Nếu quên cắt `[2:]`, tập `set('0b11')` có 3 phần tử, và mọi số đều bị coi là **không** Monobit (kể cả `3` và `7`).

In [24]:
def wrong_no_slice(n):
    return sum(1 for x in range(n + 1) if len(set(bin(x))) == 1)      # SAI: bin(x) có '0b'

print(wrong_no_slice(4), "!=", Solution().countMonobit(4))
assert wrong_no_slice(4) == 0 and Solution().countMonobit(4) == 3

0 != 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "mọi bit giống nhau" -> nghĩ ngay `0` và `2^k - 1`.
- `x & (x + 1) == 0` là mẹo "x chỉ gồm các bit 1 liên tiếp từ bit thấp" (cũng là kiểm tra `x + 1` là lũy thừa của 2).
- Khi ràng buộc nhỏ (`n <= 1000`) hãy viết brute force trước để làm "oracle", rồi mới thử công thức và so sánh.
- Luôn thử `n = 0`, `n = 2^k - 1`, `n = 2^k` để kiểm tra sai biên.
- Bảng giá trị nhỏ (`n` từ 0 đến 16) là cách nhanh nhất để đoán và kiểm chứng công thức.

## 15. Biến thể của bài

1. Đếm số có **đúng một bit 1** (lũy thừa của 2) trong `[1, n]`: LeetCode 231 (Power of Two) mở rộng.
2. **Number of 1 Bits** (191) và **Counting Bits** (338): đếm bit 1.
3. **Binary Number with Alternating Bits** (693): các bit xen kẽ.
4. **Complement of Base 10 Integer** (1009): lấy mặt nạ `(1 << bit_length) - 1` rồi XOR.
5. Phiên bản khó: đếm số có mọi bit giống nhau với `n` rất lớn (`10^18`) -> bắt buộc dùng công thức.
6. Đếm số palindrome nhị phân trong `[0, n]`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | chuỗi nhị phân + `set` | `O(n log n)` | `O(log n)` | Oracle, đúng theo đề |
| 2. Mẹo bit | `x & (x+1) == 0` | `O(n)` | `O(1)` | Duyệt nhanh gọn |
| 3. Sinh trực tiếp | `cur = 2*cur+1` | `O(log n)` | `O(1)` | Nộp bài, dễ chứng minh |
| 4. Mặt nạ | `x == (1<<bl)-1` | `O(n)` | `O(1)` | Đọc dễ hiểu |
| 5. Công thức | `(n+1).bit_length()` | `O(1)` | `O(1)` | Khi `n` rất lớn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().countMonobit(10)` và `Solution().countMonobit(15)` ra gì?

**Bài 2 (tính tay):** liệt kê các số Monobit trong `[0, 100]`.

**Bài 3 (điền TODO):** hoàn thành `list_monobit(n)` trả về **danh sách** các số Monobit trong `[0, n]`, tăng dần.

**Bài 4 (tự viết lại):** viết `count_single_bit_one(n)` đếm số trong `[1, n]` có đúng một bit `1` (lũy thừa của 2), không dùng vòng lặp qua mọi số.

**Bài 5 (mini challenge):** viết `kth_monobit(k)` trả về số Monobit thứ `k` (`k = 1` ứng với `0`, `k = 2` ứng với `1`, ...).

In [25]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().countMonobit(10), Solution().countMonobit(15))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def list_monobit_todo(n):
    result = []
    # TODO: sinh 0, 1, 3, 7, ... và dừng khi vượt n
    return result

4 5


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: dùng bảng ở mục 4: `10` giữ 4 số (`0,1,3,7`), `15` thêm số `15`.

**Mức 2 (vừa):** Bài 3: giống Cách 3 nhưng `append` thay vì tăng biến đếm. Bài 4: lũy thừa của 2 trong `[1, n]` có dạng `1, 2, 4, ...`; có bao nhiêu số `2^k <= n`? Bài 5: số thứ `k` (với `k >= 2`) là `2^(k-1) - 1`.

**Mức 3 (gần đáp án):** Bài 2: `[0, 1, 3, 7, 15, 31, 63]`. Bài 4: `n.bit_length()`. Bài 5: `0 if k == 1 else (1 << (k - 1)) - 1`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
assert Solution().countMonobit(10) == 4 and Solution().countMonobit(15) == 5
def list_monobit(n):
    result = [0]
    cur = 1
    while cur <= n:
        result.append(cur)
        cur = cur * 2 + 1
    return result
assert list_monobit(100) == [0, 1, 3, 7, 15, 31, 63]
assert list_monobit(0) == [0] and list_monobit(1) == [0, 1]

# Bài 4
def count_single_bit_one(n):
    return n.bit_length()
for n in range(1, 200):
    assert count_single_bit_one(n) == sum(1 for x in range(1, n + 1) if bin(x).count("1") == 1)

# Bài 5
def kth_monobit(k):
    return 0 if k == 1 else (1 << (k - 1)) - 1
assert [kth_monobit(k) for k in range(1, 8)] == [0, 1, 3, 7, 15, 31, 63]
for n in range(0, 300):
    assert len(list_monobit(n)) == Solution().countMonobit(n)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm số có dạng đặc biệt bằng cách sinh trực tiếp":**

```python
count = 1                 # phần tử đặc biệt (số 0)
cur = 1                   # phần tử nhỏ nhất của họ 2^k - 1
while cur <= n:
    count += 1
    cur = cur * 2 + 1
return count
# hoặc công thức: (n + 1).bit_length()
```

**Checklist trước khi submit:**

- [ ] Đã tính số `0` (`"0"` là Monobit) chưa?
- [ ] Họ số là `2^k - 1` (`1, 3, 7, 15`), không phải `2^k`?
- [ ] Điều kiện dừng dùng `<=` để tính cả `n = 2^k - 1`?
- [ ] Đã thử `n = 0`, `n = 1`, `n = 3`, `n = 4`, `n = 1000`?
- [ ] Chữ ký đúng `countMonobit(self, n: int) -> int`?